<a href="https://colab.research.google.com/github/rifahnanjiba02-arch/vision-language-composition/blob/main/scripts/MACCO_Table21_Winoground_MMVP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MACCO Table 21: Winoground and MMVP
This notebook evaluates the released **MACCO ViT-B/32** checkpoint. There is no training, Google Drive, GitHub clone, or author code download. The two datasets are downloaded directly from their publishers.

Each Winoground question has two images and two captions. The model must match both. MMVP tests whether it can distinguish pairs of images that look different to people but challenge vision models. Results below are percentages.

**Paper row to check:** Winoground text **32.2**, image **11.1**, group **8.2**; MMVP average **21.5**. The Winoground evaluation uses the **171-item clean split** listed in the MACCO evaluation code. The MMVP evaluation uses the benchmark authors' **135 image pairs**, with a point only when both matching decisions are right.


## 1. Install and load the trained model

In [1]:
%pip -q install open_clip_torch==2.32.0 huggingface_hub pandas requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.1 MB/s eta 0:00:00


In [2]:
import csv, json, tarfile, zipfile
from pathlib import Path
import numpy as np, pandas as pd, requests, torch, open_clip
from PIL import Image
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download, hf_hub_url, snapshot_download

device = 'cuda' if torch.cuda.is_available() else 'cpu'
data = Path('/content/macco_table21')
data.mkdir(parents=True, exist_ok=True)
print('Device:', device)

Device: cpu


In [3]:
checkpoint = hf_hub_download('hiker-lw/MACCO', 'MACCO-CLIP-ViT-B-32.pt')
model, _, preprocess = open_clip.create_model_and_transforms(
    'ViT-B-32', pretrained=checkpoint, device=device, precision='fp32')
model.eval()
print('Trained MACCO model loaded.')

MACCO-CLIP-ViT-B-32.pt: reconstructing file:   0%|          |  0.00B /  605MB            

MACCO-CLIP-ViT-B-32.pt: downloading bytes:           |  0.00B            

Trained MACCO model loaded.


## 2. Download Winoground and use the paper's test cases

The MACCO dataset release includes the Winoground images and captions. This reads its compressed archive directly into Colab; it does not keep another archive copy.

In [4]:
if not list(data.rglob('examples.jsonl')):
    url = hf_hub_url('hiker-lw/VL-Compositionality-Benchmarks',
                     'winoground.tar.gz', repo_type='dataset')
    with requests.get(url, stream=True, timeout=120) as response:
        response.raise_for_status()
        with tarfile.open(fileobj=response.raw, mode='r|*') as archive:
            for member in archive:
                archive.extract(member, data, filter='data')

wino = next(data.rglob('examples.jsonl')).parent
if not (wino / 'images').is_dir():
    with zipfile.ZipFile(next(wino.rglob('images.zip'))) as archive:
        archive.extractall(wino)
examples = [json.loads(line) for line in (wino / 'examples.jsonl').read_text().splitlines()]
print('Winoground examples:', len(examples))

Winoground examples: 400


In [5]:
CLEAN_IDS = {0, 1, 2, 5, 6, 7, 8, 9, 11, 12, 14, 15, 17, 18, 19, 20, 21, 24, 26, 29, 30, 32, 33, 34, 35, 37, 39, 43, 45, 47, 48, 50, 51, 52, 53, 54, 56, 57, 59, 60, 64, 66, 67, 71, 79, 80, 85, 87, 89, 90, 91, 92, 94, 98, 99, 100, 101, 102, 104, 105, 106, 107, 108, 109, 112, 115, 117, 120, 122, 123, 124, 125, 126, 127, 129, 137, 139, 140, 141, 142, 145, 146, 147, 151, 153, 154, 157, 158, 160, 161, 162, 165, 166, 167, 168, 169, 170, 171, 175, 177, 178, 179, 180, 181, 183, 184, 185, 186, 194, 195, 196, 197, 202, 205, 207, 212, 213, 216, 225, 231, 236, 240, 243, 244, 248, 250, 251, 252, 256, 259, 261, 265, 266, 269, 270, 271, 272, 273, 278, 279, 283, 285, 288, 289, 290, 291, 294, 297, 301, 302, 306, 308, 309, 317, 328, 337, 341, 349, 357, 360, 366, 368, 369, 370, 372, 378, 379, 380, 389, 391, 397}
winoground = [x for x in examples if x['id'] in CLEAN_IDS]
assert len(examples) == 400 and len(winoground) == 171
wino_pairs = [(wino/'images'/(x['image_0']+'.png'),
               wino/'images'/(x['image_1']+'.png'),
               x['caption_0'], x['caption_1']) for x in winoground]
print('MACCO clean split:', len(wino_pairs), 'pairs')

MACCO clean split: 171 pairs


## 3. Score image and caption pairs

One shared function computes the four image–caption similarities for each pair. The same function is used for both benchmarks.

In [6]:
@torch.inference_mode()
def score_pairs(pairs, batch_size=16):
    batches = []
    for start in tqdm(range(0, len(pairs), batch_size)):
        batch = pairs[start:start + batch_size]
        images = []
        for image0, image1, _, _ in batch:
            for path in (image0, image1):
                with Image.open(path) as image:
                    images.append(preprocess(image.convert('RGB')))
        captions = [c for _, _, c0, c1 in batch for c in (c0, c1)]
        image_vectors = model.encode_image(torch.stack(images).to(device)).cpu().numpy()
        text_vectors = model.encode_text(open_clip.tokenize(captions).to(device)).cpu().numpy()
        image_vectors /= np.linalg.norm(image_vectors, axis=1, keepdims=True)
        text_vectors /= np.linalg.norm(text_vectors, axis=1, keepdims=True)
        batches.append(np.einsum('nkd,nld->nkl',
            image_vectors.reshape(-1, 2, image_vectors.shape[-1]),
            text_vectors.reshape(-1, 2, text_vectors.shape[-1])))
    return np.concatenate(batches)

In [7]:
s = score_pairs(wino_pairs)
text_right = (s[:, 0, 0] > s[:, 0, 1]) & (s[:, 1, 1] > s[:, 1, 0])
image_right = (s[:, 0, 0] > s[:, 1, 0]) & (s[:, 1, 1] > s[:, 0, 1])
wino_scores = [100 * x.mean() for x in (text_right, image_right, text_right & image_right)]
print('Winoground text, image, group:', [round(x, 2) for x in wino_scores])

  0%|          | 0/11 [00:00<?, ?it/s]

Winoground text, image, group: [np.float64(32.16), np.float64(11.11), np.float64(8.19)]


## 4. Download and evaluate MMVP-VLM

The original MMVP-VLM release has 270 images arranged into 135 pairs. This uses its provided captions and requires both decisions in a pair to be right.

In [8]:
mmvp = Path(snapshot_download('MMVP/MMVP_VLM', repo_type='dataset',
    allow_patterns=['Questions.csv', 'MLLM_VLM Images/**']))
with (mmvp / 'Questions.csv').open(newline='') as file:
    questions = list(csv.DictReader(file))

mmvp_pairs = []
for a, b in zip(questions[::2], questions[1::2]):
    first, second = int(a['Question ID']), int(b['Question ID'])
    assert first % 2 == 1 and second == first + 1 and a['Type'] == b['Type']
    folder = mmvp / 'MLLM_VLM Images' / a['Type']
    mmvp_pairs.append((folder / f'{first}.jpg', folder / f'{second}.jpg',
                       'a photo of ' + a['Statement'], 'a photo of ' + b['Statement']))
assert len(questions) == 270 and len(mmvp_pairs) == 135
print('MMVP pairs:', len(mmvp_pairs))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 271 files:   0%|          | 0/271 [00:00<?, ?it/s]

MMVP pairs: 135


In [9]:
m = score_pairs(mmvp_pairs)
mmvp_right = (m[:, 0, 0] > m[:, 1, 0]) & (m[:, 1, 1] > m[:, 0, 1])
mmvp_score = 100 * mmvp_right.mean()
print('MMVP pair accuracy:', round(mmvp_score, 2))

  0%|          | 0/9 [00:00<?, ?it/s]

MMVP pair accuracy: 21.48


## 5. Compare with Table 21

The first three rows are **copied reference values from the paper**. Only the final row comes from the model evaluation above.

In [10]:
columns = ['Winoground Text', 'Winoground Image', 'Winoground Group', 'MMVP Avg.']
table = pd.DataFrame([
    [31.6, 11.1, 9.4, 14.8],
    [32.2, 8.8, 5.9, 20.7],
    [32.2, 11.1, 8.2, 21.5],
    [*wino_scores, mmvp_score],
], index=['CLIP (paper)', 'CLIP-FT (paper)', 'MACCO (paper)', 'MACCO (ours)'],
   columns=columns)
display(table.round(2))
table.to_csv('/content/MACCO_Table21.csv')

,Winoground Text,Winoground Image,Winoground Group,MMVP Avg.
CLIP (paper),31.60,11.10,9.40,14.80
CLIP-FT (paper),32.20,8.80,5.90,20.70
MACCO (paper),32.20,11.10,8.20,21.50
MACCO (ours),32.16,11.11,8.19,21.48


In [11]:
from google.colab import files
files.download('/content/MACCO_Table21.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Sources:** [MACCO paper, Table 21](https://aclanthology.org/2026.acl-long.1490.pdf) · [Released MACCO checkpoint](https://huggingface.co/hiker-lw/MACCO) · [MACCO Winoground data](https://huggingface.co/datasets/hiker-lw/VL-Compositionality-Benchmarks/tree/main) · [MMVP-VLM dataset](https://huggingface.co/datasets/MMVP/MMVP_VLM) · [MMVP evaluation rule](https://github.com/tsb0601/MMVP/blob/main/scripts/evaluate_vlm.py)

This notebook evaluates the released MACCO checkpoint only. The CLIP and CLIP-FT rows are paper reference numbers, not measurements made here.